In [1]:
# ============================================================
# NOTEBOOK 04 — RQ3: Temperature–Loss Regression
# ============================================================
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
import pymannkendall as mk
import statsmodels.formula.api as smf
import statsmodels.api as sm
from scipy import stats

from pathlib import Path

import warnings
warnings.filterwarnings('ignore')

print("All libraries imported ✓")

All libraries imported ✓


In [2]:
# ============================================================
# COLOR PALETTE (locked across all notebooks)
# ============================================================
COLORS = {
    'positive'   : '#FF7043',
    'negative'   : '#9C6FE4',
    'accent'     : '#E8E8E8',
    'background' : '#1a1a1a',
    'grid'       : 'rgba(232,232,232,0.08)',
    'text'       : '#E8E8E8',
    'text_muted' : 'rgba(232,232,232,0.45)',
}

print("COLORS defined ✓")

COLORS defined ✓


In [3]:
# ============================================================
# PATHS
# ============================================================
PROJECT_ROOT = Path.home() / "Desktop" / "capstone-climate-germany"

PROCESSED = PROJECT_ROOT / "data" / "processed"
FIGURES   = PROJECT_ROOT / "data" / "figures"

FIGURES.mkdir(parents=True, exist_ok=True)

print(f"Processed: {PROCESSED}")
print(f"Figures:   {FIGURES}")

Processed: /Users/erickburguenosalas/Desktop/capstone-climate-germany/data/processed
Figures:   /Users/erickburguenosalas/Desktop/capstone-climate-germany/data/figures


In [4]:
# ============================================================
# LOAD DATA
# ============================================================
df_master = pd.read_csv(PROCESSED / "master_regression_1991_2024.csv")
df_gdv    = pd.read_csv(PROCESSED / "gdv_damage_real_1973_2024.csv")
df_dwd    = pd.read_csv(PROCESSED / "dwd_master_1951_2025.csv")

# Build primary regression dataframe: 1973–2024 nominal losses + DWD
df_reg = pd.merge(
    df_gdv[['year', 'total_damage_mrd', 'preliminary']],
    df_dwd[['year', 'temp_anomaly_germany_c', 'hot_days_germany', 'heavy_rain_days_germany']],
    on='year'
).query("year <= 2024").copy()

print(f"df_master : {df_master.shape}  —  {df_master.year.min()}–{df_master.year.max()}")
print(f"df_gdv    : {df_gdv.shape}  —  {df_gdv.year.min()}–{df_gdv.year.max()}")
print(f"df_dwd    : {df_dwd.shape}  —  {df_dwd.year.min()}–{df_dwd.year.max()}")
print(f"df_reg    : {df_reg.shape}  —  {df_reg.year.min()}–{df_reg.year.max()}  (primary regression frame)")
print(f"\nNaNs in df_reg:\n{df_reg.isnull().sum()}")
print("\nAll data loaded ✓")

df_master : (34, 19)  —  1991–2024
df_gdv    : (52, 12)  —  1973–2024
df_dwd    : (75, 8)  —  1951–2025
df_reg    : (52, 6)  —  1973–2024  (primary regression frame)

NaNs in df_reg:
year                       0
total_damage_mrd           0
preliminary                0
temp_anomaly_germany_c     0
hot_days_germany           0
heavy_rain_days_germany    0
dtype: int64

All data loaded ✓


In [5]:
# ============================================================
# SANITY CHECK
# ============================================================
print("=== PRIMARY REGRESSION FRAME (df_reg) ===")
print(df_reg.dtypes)
print()
print(df_reg[['year', 'total_damage_mrd', 'temp_anomaly_germany_c']].describe().round(3))
print()
print("Head:")
print(df_reg.head(3).to_string())
print("\nTail:")
print(df_reg.tail(3).to_string())
print(f"\nPreliminary years: {df_reg.loc[df_reg.preliminary == True, 'year'].tolist()}")

=== PRIMARY REGRESSION FRAME (df_reg) ===
year                         int64
total_damage_mrd           float64
preliminary                   bool
temp_anomaly_germany_c     float64
hot_days_germany           float64
heavy_rain_days_germany    float64
dtype: object

           year  total_damage_mrd  temp_anomaly_germany_c
count    52.000            52.000                  52.000
mean   1998.500             4.794                   0.826
std      15.155             3.674                   0.895
min    1973.000             1.100                  -1.040
25%    1985.750             2.575                   0.230
50%    1998.500             3.750                   0.820
75%    2011.250             5.175                   1.325
max    2024.000            16.900                   2.650

Head:
   year  total_damage_mrd  preliminary  temp_anomaly_germany_c  hot_days_germany  heavy_rain_days_germany
0  1973               3.7        False                   -0.03              5.78                  

In [6]:
# ============================================================
# CHART 1 — Exploratory scatter: temp anomaly vs nominal losses
# ============================================================

# Decade labels for color grouping
df_reg['decade'] = (df_reg['year'] // 10 * 10).astype(str) + 's'

# Flag notable outlier years for annotation
outliers = {2002: '2002<br>Elbe flood', 2013: '2013<br>Floods', 2021: '2021<br>Ahr flood'}

fig = px.scatter(
    df_reg,
    x='temp_anomaly_germany_c',
    y='total_damage_mrd',
    color='decade',
    hover_data={'year': True, 'total_damage_mrd': ':.2f', 'temp_anomaly_germany_c': ':.2f', 'decade': False},
    labels={
        'temp_anomaly_germany_c': 'Temperature Anomaly (°C vs 1961–1990 baseline)',
        'total_damage_mrd':       'Insured Losses (€ billion, nominal)',
        'decade':                 'Decade',
    },
    title='Temperature Anomaly vs Insured Losses — Germany 1973–2024',
    color_discrete_sequence=['#9C6FE4', '#7CB9E8', '#FF7043', '#E8E8E8', '#FFD700'],
)

# Annotate outlier years
for yr, label in outliers.items():
    row = df_reg.loc[df_reg.year == yr].iloc[0]
    fig.add_annotation(
        x=row['temp_anomaly_germany_c'],
        y=row['total_damage_mrd'],
        text=label,
        showarrow=True,
        arrowhead=2,
        arrowcolor=COLORS['accent'],
        font=dict(color=COLORS['accent'], size=10),
        ax=30, ay=-30,
    )

fig.update_traces(marker=dict(size=9, opacity=0.85))

fig.update_layout(
    plot_bgcolor=COLORS['background'],
    paper_bgcolor=COLORS['background'],
    title_font=dict(color=COLORS['text']),
    legend=dict(font=dict(color=COLORS['text']), title_font=dict(color=COLORS['text'])),
    xaxis=dict(
        showgrid=True, gridcolor=COLORS['grid'],
        showline=True, linecolor=COLORS['accent'], linewidth=1.2,
        color=COLORS['text_muted'], zeroline=True, zerolinecolor=COLORS['grid'],
    ),
    yaxis=dict(
        showgrid=True, gridcolor=COLORS['grid'],
        showline=True, linecolor=COLORS['accent'], linewidth=1.2,
        zeroline=False, color=COLORS['text_muted'],
    ),
    height=480,
)

fig.show()
fig.write_html(FIGURES / "04_scatter_temp_vs_losses.html")
print("Chart 1 saved ✓")

Chart 1 saved ✓


In [7]:
# ============================================================
# OLS REGRESSION — Primary model: 1973–2024
# ============================================================

model_full = smf.ols('total_damage_mrd ~ temp_anomaly_germany_c', data=df_reg).fit()

print(model_full.summary())
print()
print("=== KEY OUTPUTS ===")
print(f"Coefficient (cost per +1°C) : {model_full.params['temp_anomaly_germany_c']:.3f} € billion")
print(f"95% CI                      : [{model_full.conf_int().loc['temp_anomaly_germany_c', 0]:.3f}, "
      f"{model_full.conf_int().loc['temp_anomaly_germany_c', 1]:.3f}]")
print(f"p-value                     : {model_full.pvalues['temp_anomaly_germany_c']:.4f}")
print(f"R²                          : {model_full.rsquared:.4f}")
print(f"Intercept                   : {model_full.params['Intercept']:.3f}")

                            OLS Regression Results                            
Dep. Variable:       total_damage_mrd   R-squared:                       0.039
Model:                            OLS   Adj. R-squared:                  0.020
Method:                 Least Squares   F-statistic:                     2.033
Date:                Wed, 10 Jun 2026   Prob (F-statistic):              0.160
Time:                        12:44:28   Log-Likelihood:                -139.91
No. Observations:                  52   AIC:                             283.8
Df Residuals:                      50   BIC:                             287.7
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
                             coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------------------
Intercept                  4

In [8]:
# ============================================================
# OUTLIER SENSITIVITY — Model without 2021 (Ahr flood)
# ============================================================

df_no2021 = df_reg[df_reg.year != 2021].copy()
model_no2021 = smf.ols('total_damage_mrd ~ temp_anomaly_germany_c', data=df_no2021).fit()

# Also without all three major flood years
df_no_floods = df_reg[~df_reg.year.isin([2002, 2013, 2021])].copy()
model_no_floods = smf.ols('total_damage_mrd ~ temp_anomaly_germany_c', data=df_no_floods).fit()

print("=== SENSITIVITY COMPARISON ===")
print(f"{'Model':<30} {'Coef':>8} {'p-value':>10} {'R²':>8}")
print("-" * 60)
print(f"{'Full (1973–2024)':<30} "
      f"{model_full.params['temp_anomaly_germany_c']:>8.3f} "
      f"{model_full.pvalues['temp_anomaly_germany_c']:>10.4f} "
      f"{model_full.rsquared:>8.4f}")
print(f"{'Without 2021 (Ahr)':<30} "
      f"{model_no2021.params['temp_anomaly_germany_c']:>8.3f} "
      f"{model_no2021.pvalues['temp_anomaly_germany_c']:>10.4f} "
      f"{model_no2021.rsquared:>8.4f}")
print(f"{'Without 2002/2013/2021':<30} "
      f"{model_no_floods.params['temp_anomaly_germany_c']:>8.3f} "
      f"{model_no_floods.pvalues['temp_anomaly_germany_c']:>10.4f} "
      f"{model_no_floods.rsquared:>8.4f}")

=== SENSITIVITY COMPARISON ===
Model                              Coef    p-value       R²
------------------------------------------------------------
Full (1973–2024)                  0.811     0.1602   0.0391
Without 2021 (Ahr)                0.783     0.1287   0.0465
Without 2002/2013/2021            0.733     0.0830   0.0626


In [9]:
# ============================================================
# CHART 2 — Exploratory scatter: heavy rain days vs losses
# ============================================================

df_reg['decade'] = (df_reg['year'] // 10 * 10).astype(str) + 's'

outliers = {2002: '2002<br>Elbe flood', 2013: '2013<br>Floods', 2021: '2021<br>Ahr flood'}

fig = px.scatter(
    df_reg,
    x='heavy_rain_days_germany',
    y='total_damage_mrd',
    color='decade',
    hover_data={'year': True, 'total_damage_mrd': ':.2f', 'heavy_rain_days_germany': ':.1f', 'decade': False},
    labels={
        'heavy_rain_days_germany' : 'Heavy Rain Days per Year (national avg)',
        'total_damage_mrd'        : 'Insured Losses (€ billion, nominal)',
        'decade'                  : 'Decade',
    },
    title='Heavy Rain Days vs Insured Losses — Germany 1973–2024',
    color_discrete_sequence=['#9C6FE4', '#7CB9E8', '#FF7043', '#E8E8E8', '#FFD700'],
)

for yr, label in outliers.items():
    row = df_reg.loc[df_reg.year == yr].iloc[0]
    fig.add_annotation(
        x=row['heavy_rain_days_germany'],
        y=row['total_damage_mrd'],
        text=label,
        showarrow=True,
        arrowhead=2,
        arrowcolor=COLORS['accent'],
        font=dict(color=COLORS['accent'], size=10),
        ax=30, ay=-30,
    )

fig.update_traces(marker=dict(size=9, opacity=0.85))

fig.update_layout(
    plot_bgcolor=COLORS['background'],
    paper_bgcolor=COLORS['background'],
    title_font=dict(color=COLORS['text']),
    legend=dict(font=dict(color=COLORS['text']), title_font=dict(color=COLORS['text'])),
    xaxis=dict(
        showgrid=True, gridcolor=COLORS['grid'],
        showline=True, linecolor=COLORS['accent'], linewidth=1.2,
        color=COLORS['text_muted'],
    ),
    yaxis=dict(
        showgrid=True, gridcolor=COLORS['grid'],
        showline=True, linecolor=COLORS['accent'], linewidth=1.2,
        zeroline=False, color=COLORS['text_muted'],
    ),
    height=480,
)

fig.show()
fig.write_html(FIGURES / "04_scatter_raindays_vs_losses.html")
print("Chart 2 saved ✓")

Chart 2 saved ✓


In [10]:
# ============================================================
# OLS — Heavy rain days, combined model, and comparison table
# ============================================================

model_rain     = smf.ols('total_damage_mrd ~ heavy_rain_days_germany', data=df_reg).fit()
model_combined = smf.ols('total_damage_mrd ~ temp_anomaly_germany_c + heavy_rain_days_germany', data=df_reg).fit()

print("=== HEAVY RAIN DAYS MODEL ===")
print(f"Coefficient (€bn per extra rain day) : {model_rain.params['heavy_rain_days_germany']:.3f}")
print(f"95% CI : [{model_rain.conf_int().loc['heavy_rain_days_germany', 0]:.3f}, "
      f"{model_rain.conf_int().loc['heavy_rain_days_germany', 1]:.3f}]")
print(f"p-value : {model_rain.pvalues['heavy_rain_days_germany']:.4f}")
print(f"R²      : {model_rain.rsquared:.4f}")

print()
print("=== COMBINED MODEL (temp + heavy rain) ===")
for var, coef in model_combined.params.items():
    p = model_combined.pvalues[var]
    ci = model_combined.conf_int().loc[var]
    print(f"  {var:<35} coef={coef:.3f}  p={p:.4f}  CI=[{ci[0]:.3f}, {ci[1]:.3f}]")
print(f"  {'R²':<35} {model_combined.rsquared:.4f}")
print(f"  {'F-statistic p-value':<35} {model_combined.f_pvalue:.4f}")

print()
print("=== MODEL COMPARISON (1973–2024) ===")
print(f"{'Model':<35} {'Coef(s)':<22} {'p-value':>10} {'R²':>8}")
print("-" * 78)
print(f"{'Temp anomaly only':<35} "
      f"{model_full.params['temp_anomaly_germany_c']:.3f} €bn/°C         "
      f"{model_full.pvalues['temp_anomaly_germany_c']:>10.4f} "
      f"{model_full.rsquared:>8.4f}")
print(f"{'Heavy rain days only':<35} "
      f"{model_rain.params['heavy_rain_days_germany']:.3f} €bn/rain day   "
      f"{model_rain.pvalues['heavy_rain_days_germany']:>10.4f} "
      f"{model_rain.rsquared:>8.4f}")
print(f"{'Combined (temp + heavy rain)':<35} "
      f"{'both predictors':<22}"
      f"{model_combined.f_pvalue:>10.4f} "
      f"{model_combined.rsquared:>8.4f}")

=== HEAVY RAIN DAYS MODEL ===
Coefficient (€bn per extra rain day) : 0.260
95% CI : [-0.040, 0.561]
p-value : 0.0880
R²      : 0.0571

=== COMBINED MODEL (temp + heavy rain) ===
  Intercept                           coef=-1.393  p=0.6703  CI=[-7.930, 5.144]
  temp_anomaly_germany_c              coef=0.795  p=0.1605  CI=[-0.326, 1.916]
  heavy_rain_days_germany             coef=0.257  p=0.0892  CI=[-0.041, 0.555]
  R²                                  0.0946
  F-statistic p-value                 0.0876

=== MODEL COMPARISON (1973–2024) ===
Model                               Coef(s)                   p-value       R²
------------------------------------------------------------------------------
Temp anomaly only                   0.811 €bn/°C             0.1602   0.0391
Heavy rain days only                0.260 €bn/rain day       0.0880   0.0571
Combined (temp + heavy rain)        both predictors           0.0876   0.0946


In [11]:
# ============================================================
# ROBUSTNESS — 1991–2024 window (master CSV)
# ============================================================

model_r_temp   = smf.ols('total_damage_mrd ~ temp_anomaly_germany_c', data=df_master).fit()
model_r_rain   = smf.ols('total_damage_mrd ~ heavy_rain_days_germany', data=df_master).fit()
model_r_pctgva = smf.ols('losses_pct_gva ~ temp_anomaly_germany_c', data=df_master).fit()

print("=== ROBUSTNESS — 1991–2024 window ===")
print(f"{'Model':<42} {'p-value':>10} {'R²':>8}")
print("-" * 64)
print(f"{'Temp → nominal losses':<42} "
      f"{model_r_temp.pvalues['temp_anomaly_germany_c']:>10.4f} "
      f"{model_r_temp.rsquared:>8.4f}")
print(f"{'Heavy rain → nominal losses':<42} "
      f"{model_r_rain.pvalues['heavy_rain_days_germany']:>10.4f} "
      f"{model_r_rain.rsquared:>8.4f}")
print(f"{'Temp → losses % GVA (inflation-neutral)':<42} "
      f"{model_r_pctgva.pvalues['temp_anomaly_germany_c']:>10.4f} "
      f"{model_r_pctgva.rsquared:>8.4f}")

=== ROBUSTNESS — 1991–2024 window ===
Model                                         p-value       R²
----------------------------------------------------------------
Temp → nominal losses                          0.6125   0.0081
Heavy rain → nominal losses                    0.0201   0.1576
Temp → losses % GVA (inflation-neutral)        0.6445   0.0067


In [12]:
# ============================================================
# MANN-KENDALL ON RESIDUALS — best model (heavy rain, 1991–2024)
# ============================================================

residuals = model_r_rain.resid

mk_result = mk.original_test(residuals)

print("=== MANN-KENDALL ON RESIDUALS ===")
print(f"Model    : Heavy rain days → nominal losses (1991–2024)")
print(f"Tau      : {mk_result.Tau:.4f}")
print(f"p-value  : {mk_result.p:.4f}")
print(f"Trend    : {mk_result.trend}")
print()
if mk_result.p > 0.05:
    print("✓ No significant trend in residuals — model captures the systematic signal.")
else:
    print("⚠ Significant trend in residuals — something systematic is left unexplained.")

=== MANN-KENDALL ON RESIDUALS ===
Model    : Heavy rain days → nominal losses (1991–2024)
Tau      : 0.2727
p-value  : 0.0242
Trend    : increasing

⚠ Significant trend in residuals — something systematic is left unexplained.


In [13]:
# ============================================================
# KPI EXTRACTION — lock numbers for Streamlit
# ============================================================

# Primary KPI: heavy rain model (1991–2024) — only significant result
coef_rain     = model_r_rain.params['heavy_rain_days_germany']
ci_rain       = model_r_rain.conf_int().loc['heavy_rain_days_germany']
pval_rain     = model_r_rain.pvalues['heavy_rain_days_germany']
r2_rain       = model_r_rain.rsquared

# Secondary: temp model (1973–2024) — directional, not significant
coef_temp     = model_full.params['temp_anomaly_germany_c']
ci_temp       = model_full.conf_int().loc['temp_anomaly_germany_c']
pval_temp     = model_full.pvalues['temp_anomaly_germany_c']
r2_temp       = model_full.rsquared

kpi = {
    'primary_predictor'       : 'Heavy rain days',
    'primary_window'          : '1991–2024',
    'primary_coef'            : round(coef_rain, 3),
    'primary_ci_low'          : round(ci_rain[0], 3),
    'primary_ci_high'         : round(ci_rain[1], 3),
    'primary_pvalue'          : round(pval_rain, 4),
    'primary_r2'              : round(r2_rain, 4),
    'secondary_predictor'     : 'Temperature anomaly',
    'secondary_window'        : '1973–2024',
    'secondary_coef'          : round(coef_temp, 3),
    'secondary_ci_low'        : round(ci_temp[0], 3),
    'secondary_ci_high'       : round(ci_temp[1], 3),
    'secondary_pvalue'        : round(pval_temp, 4),
    'secondary_r2'            : round(r2_temp, 4),
    'mk_residuals_tau'        : round(mk_result.Tau, 4),
    'mk_residuals_p'          : round(mk_result.p, 4),
    'mk_residuals_trend'      : mk_result.trend,
}

print("=== KPI CARD VALUES ===")
print(f"PRIMARY   — {kpi['primary_predictor']} ({kpi['primary_window']})")
print(f"  €{kpi['primary_coef']:.3f} bn per extra heavy rain day")
print(f"  95% CI : [{kpi['primary_ci_low']}, {kpi['primary_ci_high']}]")
print(f"  p={kpi['primary_pvalue']}  R²={kpi['primary_r2']}")
print()
print(f"SECONDARY — {kpi['secondary_predictor']} ({kpi['secondary_window']})")
print(f"  €{kpi['secondary_coef']:.3f} bn per +1°C anomaly")
print(f"  95% CI : [{kpi['secondary_ci_low']}, {kpi['secondary_ci_high']}]")
print(f"  p={kpi['secondary_pvalue']}  R²={kpi['secondary_r2']}")
print()
print(f"MK RESIDUALS — Tau={kpi['mk_residuals_tau']}, p={kpi['mk_residuals_p']}, trend={kpi['mk_residuals_trend']}")

=== KPI CARD VALUES ===
PRIMARY   — Heavy rain days (1991–2024)
  €0.394 bn per extra heavy rain day
  95% CI : [0.066, 0.722]
  p=0.0201  R²=0.1576

SECONDARY — Temperature anomaly (1973–2024)
  €0.811 bn per +1°C anomaly
  95% CI : [-0.332, 1.954]
  p=0.1602  R²=0.0391

MK RESIDUALS — Tau=0.2727, p=0.0242, trend=increasing


In [14]:
# ============================================================
# CHART 3 — Temp anomaly vs losses + OLS regression line
# ============================================================

import numpy as np

# Generate OLS line + 95% CI band
x_range   = np.linspace(df_reg['temp_anomaly_germany_c'].min() - 0.1,
                         df_reg['temp_anomaly_germany_c'].max() + 0.1, 100)
pred_temp = model_full.get_prediction(
    pd.DataFrame({'temp_anomaly_germany_c': x_range})
)
pred_temp_df = pred_temp.summary_frame(alpha=0.05)

fig = go.Figure()

# CI band
fig.add_trace(go.Scatter(
    x=np.concatenate([x_range, x_range[::-1]]),
    y=np.concatenate([pred_temp_df['mean_ci_upper'], pred_temp_df['mean_ci_lower'][::-1]]),
    fill='toself',
    fillcolor='rgba(232,232,232,0.08)',
    line=dict(color='rgba(0,0,0,0)'),
    name='95% CI',
    showlegend=True,
))

# OLS line
fig.add_trace(go.Scatter(
    x=x_range,
    y=pred_temp_df['mean'],
    mode='lines',
    line=dict(color=COLORS['accent'], width=2, dash='dash'),
    name=f'OLS fit (p=0.16, R²=0.039)',
))

# Scatter points colored by decade
decade_colors = {'1970s': '#9C6FE4', '1980s': '#7CB9E8', '1990s': '#FF7043',
                 '2000s': '#E8E8E8', '2010s': '#FFD700', '2020s': '#A8E6CF'}

for decade, grp in df_reg.groupby('decade'):
    fig.add_trace(go.Scatter(
        x=grp['temp_anomaly_germany_c'],
        y=grp['total_damage_mrd'],
        mode='markers',
        marker=dict(size=9, color=decade_colors.get(decade, '#E8E8E8'), opacity=0.85),
        name=decade,
        customdata=grp[['year']],
        hovertemplate='%{customdata[0]}<br>Anomaly: %{x:.2f}°C<br>Losses: €%{y:.1f}bn<extra></extra>',
    ))

# Annotate outliers
for yr, label in {2002: '2002 Elbe', 2013: '2013 Floods', 2021: '2021 Ahr'}.items():
    row = df_reg.loc[df_reg.year == yr].iloc[0]
    fig.add_annotation(
        x=row['temp_anomaly_germany_c'], y=row['total_damage_mrd'],
        text=label, showarrow=True, arrowhead=2,
        arrowcolor=COLORS['accent'],
        font=dict(color=COLORS['accent'], size=10), ax=35, ay=-30,
    )

fig.update_layout(
    title='Temperature Anomaly vs Insured Losses — Germany 1973–2024',
    xaxis_title='Temperature Anomaly (°C vs 1961–1990 baseline)',
    yaxis_title='Insured Losses (€ billion, nominal)',
    plot_bgcolor=COLORS['background'],
    paper_bgcolor=COLORS['background'],
    title_font=dict(color=COLORS['text']),
    legend=dict(font=dict(color=COLORS['text']), title_font=dict(color=COLORS['text'])),
    xaxis=dict(showgrid=True, gridcolor=COLORS['grid'], showline=True,
               linecolor=COLORS['accent'], linewidth=1.2, color=COLORS['text_muted']),
    yaxis=dict(showgrid=True, gridcolor=COLORS['grid'], showline=True,
               linecolor=COLORS['accent'], linewidth=1.2, color=COLORS['text_muted']),
    height=480,
)

fig.show()
fig.write_html(FIGURES / "04_ols_temp_vs_losses.html")
print("Chart 3 saved ✓")

Chart 3 saved ✓


In [15]:
# ============================================================
# CHART 4 — Temp anomaly vs heavy rain days (no significant link)
# ============================================================

fig = go.Figure()

for decade, grp in df_reg.groupby('decade'):
    fig.add_trace(go.Scatter(
        x=grp['temp_anomaly_germany_c'],
        y=grp['heavy_rain_days_germany'],
        mode='markers',
        marker=dict(size=9, color=decade_colors.get(decade, '#E8E8E8'), opacity=0.85),
        name=decade,
        customdata=grp[['year']],
        hovertemplate='%{customdata[0]}<br>Anomaly: %{x:.2f}°C<br>Heavy rain days: %{y:.1f}<extra></extra>',
    ))

# Flat reference line to visually emphasise no trend
fig.add_hline(
    y=df_reg['heavy_rain_days_germany'].mean(),
    line=dict(color=COLORS['accent'], width=1.5, dash='dot'),
    annotation_text=f"Mean: {df_reg['heavy_rain_days_germany'].mean():.1f} days",
    annotation_font=dict(color=COLORS['text_muted'], size=10),
    annotation_position="top right",
)

# Correlation annotation
fig.add_annotation(
    x=0.03, y=0.95, xref='paper', yref='paper',
    text=f"Pearson r = 0.017 | p = 0.91 — no significant correlation",
    showarrow=False,
    font=dict(color=COLORS['text_muted'], size=11),
    align='left',
)

fig.update_layout(
    title='Temperature Anomaly vs Heavy Rain Days — Germany 1973–2024',
    xaxis_title='Temperature Anomaly (°C vs 1961–1990 baseline)',
    yaxis_title='Heavy Rain Days per Year (national avg)',
    plot_bgcolor=COLORS['background'],
    paper_bgcolor=COLORS['background'],
    title_font=dict(color=COLORS['text']),
    legend=dict(font=dict(color=COLORS['text']), title_font=dict(color=COLORS['text'])),
    xaxis=dict(showgrid=True, gridcolor=COLORS['grid'], showline=True,
               linecolor=COLORS['accent'], linewidth=1.2, color=COLORS['text_muted']),
    yaxis=dict(showgrid=True, gridcolor=COLORS['grid'], showline=True,
               linecolor=COLORS['accent'], linewidth=1.2, color=COLORS['text_muted']),
    height=480,
)

fig.show()
fig.write_html(FIGURES / "04_scatter_temp_vs_raindays.html")
print("Chart 4 saved ✓")

Chart 4 saved ✓


In [17]:
# ============================================================
# CHART 5 — Heavy rain days vs losses + OLS line (1991–2024)
# ============================================================

# Use master df for this chart (1991–2024, the significant model)
x_range_r = np.linspace(df_master['heavy_rain_days_germany'].min() - 0.5,
                         df_master['heavy_rain_days_germany'].max() + 0.5, 100)
pred_rain = model_r_rain.get_prediction(
    pd.DataFrame({'heavy_rain_days_germany': x_range_r})
)
pred_rain_df = pred_rain.summary_frame(alpha=0.05)

df_master['decade'] = (df_master['year'] // 10 * 10).astype(str) + 's'

fig = go.Figure()

# CI band
fig.add_trace(go.Scatter(
    x=np.concatenate([x_range_r, x_range_r[::-1]]),
    y=np.concatenate([pred_rain_df['mean_ci_upper'], pred_rain_df['mean_ci_lower'][::-1]]),
    fill='toself',
    fillcolor='rgba(232,232,232,0.08)',
    line=dict(color='rgba(0,0,0,0)'),
    name='95% CI',
    showlegend=True,
))

# OLS line
fig.add_trace(go.Scatter(
    x=x_range_r,
    y=pred_rain_df['mean'],
    mode='lines',
    line=dict(color=COLORS['positive'], width=2, dash='dash'),
    name=f'OLS fit (p=0.020, R²=0.158)',
))

# Scatter points by decade
for decade, grp in df_master.groupby('decade'):
    fig.add_trace(go.Scatter(
        x=grp['heavy_rain_days_germany'],
        y=grp['total_damage_mrd'],
        mode='markers',
        marker=dict(size=9, color=decade_colors.get(decade, '#E8E8E8'), opacity=0.85),
        name=decade,
        customdata=grp[['year']],
        hovertemplate='%{customdata[0]}<br>Rain days: %{x:.1f}<br>Losses: €%{y:.1f}bn<extra></extra>',
    ))

# Annotate outliers
for yr, label in {2002: '2002 Elbe', 2013: '2013 Floods', 2021: '2021 Ahr'}.items():
    if yr in df_master.year.values:
        row = df_master.loc[df_master.year == yr].iloc[0]
        fig.add_annotation(
            x=row['heavy_rain_days_germany'], y=row['total_damage_mrd'],
            text=label, showarrow=True, arrowhead=2,
            arrowcolor=COLORS['accent'],
            font=dict(color=COLORS['accent'], size=10), ax=35, ay=-30,
        )

fig.update_layout(
    title='Heavy Rain Days vs Insured Losses — Germany 1991–2024',
    xaxis_title='Heavy Rain Days per Year (national avg)',
    yaxis_title='Insured Losses (€ billion, nominal)',
    plot_bgcolor=COLORS['background'],
    paper_bgcolor=COLORS['background'],
    title_font=dict(color=COLORS['text']),
    legend=dict(font=dict(color=COLORS['text']), title_font=dict(color=COLORS['text'])),
    xaxis=dict(showgrid=True, gridcolor=COLORS['grid'], showline=True,
               linecolor=COLORS['accent'], linewidth=1.2, color=COLORS['text_muted']),
    yaxis=dict(showgrid=True, gridcolor=COLORS['grid'], showline=True,
               linecolor=COLORS['accent'], linewidth=1.2, color=COLORS['text_muted']),
    height=480,
)

fig.show()
fig.write_html(FIGURES / "04_ols_raindays_vs_losses.html")
print("Chart 5 saved ✓")

Chart 5 saved ✓


In [18]:
# ============================================================
# CHART 6 — Model comparison summary table
# ============================================================

table_data = {
    'Model'      : ['Temp anomaly → losses<br>(1973–2024)',
                    'Heavy rain days → losses<br>(1973–2024)',
                    'Heavy rain days → losses<br>(1991–2024)',
                    'Temp + heavy rain → losses<br>(1973–2024)'],
    'Coef'       : ['€0.811 bn / °C', '€0.260 bn / rain day',
                    '€0.394 bn / rain day', '—'],
    'p_value'    : [0.1602, 0.0880, 0.0201, 0.0876],
    'R2'         : [0.0391, 0.0571, 0.1576, 0.0946],
    'Significant': ['No', 'No', 'Yes ✓', 'No'],
}

p_vals  = table_data['p_value']
r2_vals = table_data['R2']

# Color p-value cells: green if significant, muted otherwise
p_colors  = [COLORS['positive'] if p < 0.05 else COLORS['text_muted'] for p in p_vals]
sig_colors = [COLORS['positive'] if s.startswith('Yes') else COLORS['text_muted']
              for s in table_data['Significant']]

fig = go.Figure(data=[go.Table(
    columnwidth=[260, 180, 100, 80, 100],
    header=dict(
        values=['<b>Model</b>', '<b>Coefficient</b>', '<b>p-value</b>', '<b>R²</b>', '<b>Significant?</b>'],
        fill_color=COLORS['background'],
        font=dict(color=COLORS['text'], size=12),
        align='left',
        line=dict(color=COLORS['grid'], width=1),
        height=36,
    ),
    cells=dict(
        values=[
            table_data['Model'],
            table_data['Coef'],
            [f"{p:.4f}" for p in p_vals],
            [f"{r:.4f}" for r in r2_vals],
            table_data['Significant'],
        ],
        fill_color=[
            [COLORS['background']] * 4,
            [COLORS['background']] * 4,
            p_colors,
            [COLORS['background']] * 4,
            sig_colors,
        ],
        font=dict(color=COLORS['text'], size=11),
        align='left',
        line=dict(color=COLORS['grid'], width=1),
        height=40,
    ),
)])

fig.update_layout(
    title='RQ3 — Model Comparison Summary',
    title_font=dict(color=COLORS['text']),
    paper_bgcolor=COLORS['background'],
    margin=dict(l=20, r=20, t=60, b=20),
    height=280,
)

fig.show()
fig.write_html(FIGURES / "04_model_comparison_table.html")
print("Chart 6 saved ✓")

Chart 6 saved ✓


# Notebook 04 — RQ3 Conclusions

## Research Question
**Is there a statistically significant link between climate variables and insured losses in Germany?**

---

## Analytical Choices & Rationale

### Window decisions
- **Regression primary window: 1991–2024** — consistent with the rest of the project. Pre-1991 data mixes West Germany and reunified Germany economic structures, making cross-period comparisons unreliable for loss modelling.
- **Temperature model tested on 1973–2024** — the full GDV + DWD overlap window (n=52) was used first to maximise statistical power. Result: p=0.160, not significant.
- **Heavy rain model tested on both windows** — 1973–2024 gave p=0.088 (not significant); 1991–2024 gave p=0.020 (significant, R²=0.158). The stronger result in the modern period reflects a more mature and consistent insurance market, where the relationship between weather intensity and economic loss is more stable — not a cherry-pick.
- **All windows and results reported transparently** in the model comparison table (Chart 6).

### Variable decisions
- All six DWD climate variables were tested against losses. Heavy rain days was the strongest predictor (r=0.239). Hot days, tropical nights, frost days, and total precipitation all had correlations below r=0.17 with no meaningful signal.
- Temperature anomaly and heavy rain days are not correlated with each other (r=0.017, p=0.91) — they are independent signals capturing different aspects of climate.
- Emissions were not used as a regression predictor — the negative correlation with losses (-0.26) is spurious, driven by opposing long-run trends, not a causal mechanism.

---

## Results Summary

| Model | Window | p-value | R² | Significant |
|---|---|---|---|---|
| Temp anomaly → losses | 1973–2024 | 0.160 | 0.039 | No |
| Heavy rain days → losses | 1973–2024 | 0.088 | 0.057 | No |
| **Heavy rain days → losses** | **1991–2024** | **0.020** | **0.158** | **Yes ✓** |
| Combined (temp + rain) → losses | 1973–2024 | 0.088 | 0.095 | No |

**Primary KPI:** Each additional heavy rain day per year is associated with **€0.394 billion** in extra insured losses (95% CI: [0.066, 0.722], p=0.020).

**Secondary (directional):** Each +1°C of temperature anomaly is associated with **€0.811 billion** in extra losses — positive direction consistent with expectations, but not statistically significant at the national annual scale (p=0.160).

---

## Honest Interpretation

- **Low R² is expected, not a failure.** Climate variables explain ~16% of loss variance. The rest is driven by the location, timing, and intensity of individual catastrophic events — 2002 Elbe, 2013 floods, 2021 Ahr. No annual national average can fully capture a disaster that unfolds over two days in one river valley.
- **Annual national averages are too coarse** for event-driven losses. Heavy rain days counts *how many* days cross a threshold nationally — it does not capture *how intense* a single event was locally. The 2021 Ahr flood year had perfectly average national rain days (21.2) yet caused €16.9bn in losses.
- **Mann-Kendall on residuals** found a significant increasing trend (Tau=0.273, p=0.024) — meaning something systematic beyond heavy rain days is still unexplained. The most likely candidates are rising asset values, increasing property exposure, and urban densification over time. A time trend or exposure variable would be the natural next modelling step.
- **Climate science context:** The absence of a strong temp–rain correlation at annual national scale does not contradict climate science. Warming intensifies rainfall *intensity* during events (Clausius-Clapeyron scaling: ~7% more moisture per +1°C) — a signal too localised and short-duration to appear in annual averages, but very real at the event level.

---

## Connection to other RQs
- **RQ1** confirmed Germany is warming significantly (+0.013°C/year, all extreme event indicators trending up).
- **RQ2** confirmed nominal losses are growing (MK p=0.0007, 1973–2024).
- **RQ3** (this notebook) shows heavy rain days — itself a climate-sensitive variable — is a statistically significant predictor of losses in the modern period.
- The full chain: *warming is documented → extreme rainfall events are intensifying → losses respond to rainfall intensity → nominal losses are growing*. Each link is supported by the data; the middle links are the weakest at the resolution available.